# Amazon Review Sentiment Analysis


### Project objective
Build a three-class sentiment classifier for Amazon product reviews using rating-derived labels, text preprocessing, TF-IDF features, and five machine-learning algorithms. The notebook compares models with accuracy, weighted precision, weighted recall, weighted F1-score, and class-level classification reports.

**Label rule:** ratings 1–2 → Negative; rating 3 → Neutral; ratings 4–5 → Positive.

**Important:** the model learns labels inferred from star ratings, not manually annotated sentiment. Neutral performance should be reviewed carefully because that class may be harder to predict.


### 01. Exploratory Data Analysis (EDA)

#### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from PIL import Image

# pip install wordcloud
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import nltk
# nltk.download('punkt_tab')
# nltk.download('stopwords')
# nltk.download('wordnet')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

#### Load the Dataset

In [ ]:
from pathlib import Path

possible_files = [Path("dataset.xlsx"), Path("dataset(1).xlsx")]
dataset_path = next((path for path in possible_files if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Dataset not found. Place dataset.xlsx or dataset(1).xlsx beside this notebook.")

df = pd.read_excel(dataset_path, sheet_name="Sheet1")
print("Loaded dataset:", dataset_path.name)
df.head()


In [ ]:
print("Dataset Shape:",df.shape)
df.info()

#### Check Null and Duplicate values

In [ ]:
# Missing values
df.isnull().sum()

In [ ]:
# Duplicate rows
print(df.duplicated().sum())

In [ ]:
# Rating distribution
print(df["rating"].value_counts().sort_index())

#### Creating sentiment labels

In [ ]:
def rating_to_sentiment(rating):
    if rating<=2:
        return 'Negative'
    elif rating==3:
        return 'Neutral'
    else:
        return 'Positive'

df['sentiment'] = df['rating'].apply(rating_to_sentiment)

df[['rating','sentiment']].head()

In [ ]:
print(df['sentiment'].value_counts())
print(df["sentiment"].value_counts(normalize=True) * 100)

#### Combining title and review text

In [ ]:
df["review"] = df['title'].astype(str)+ " "+ df['body'].astype(str)

df[['review','sentiment']].head()

#### Explore rating and sentiment distribution 

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df,x='rating',palette='coolwarm',legend=False)
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(
    data=df,
    x="sentiment",
    order=["Negative", "Neutral", "Positive"],
    hue="sentiment",
    palette={"Negative": "tomato", "Neutral": "orange", "Positive": "#32CD32"},
    legend=False
)
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.show()

#### Review length analysis

In [ ]:
df["review_length"] = df["review"].apply(len)
df["word_count"] = df["review"].apply(lambda text: len(text.split()))

df[["review_length", "word_count"]].describe().T

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="sentiment", y="word_count",
            order=["Negative", "Neutral", "Positive"])
plt.title("Review Word Count by Sentiment")
plt.show()

In [ ]:
# Viewing examples from each sentiment
for sentiment in ["Negative", "Neutral", "Positive"]:
    print(f"\n--- {sentiment} Reviews ---")
    display(df[df["sentiment"] == sentiment][["rating", "review"]].head(3))

### 02. Preprocessing

#### Clean Text for modeling

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # Keep letters, numbers, and spaces
    text = re.sub(r"\s+", " ", text).strip()     # remove extra spaces    
    return text

df["clean_review"] = df["review"].apply(clean_text)

df[["review", "clean_review"]].head()

In [ ]:
stop_words = set(stopwords.words('english'))
negation_words = {"no", "not", "never", "neither", "dont", "nor"}
stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()

# Regex for important mobile specs
spec_pattern = re.compile(r"^\d+(mAh|hz|mp|gb|tb|w|inch|ghz|nm|fps|ppi|nits)$", re.IGNORECASE)

def preprocess_text(text):
    tokens = word_tokenize(text)
    cleaned_tokens = []

    for word in tokens:
        word_lower = word.lower()

        # Keep alphabetic words (lemmatized)
        if word.isalpha() and word_lower not in stop_words:
            cleaned_tokens.append(lemmatizer.lemmatize(word_lower))

        # Keep important specs (numbers + units)
        elif spec_pattern.match(word_lower):
            cleaned_tokens.append(word_lower)

        # Keep small rating numbers (like 1–10 stars)
        elif word.isdigit() and 1 <= int(word) <= 10:
            cleaned_tokens.append(word_lower)


        # Ignore other meaningless numbers
        else:
            continue

    return " ".join(cleaned_tokens)

df["cleaned_review"] = df["clean_review"].apply(preprocess_text)

In [ ]:
# Remove empty reviews
df = df[df["cleaned_review"].str.len() > 0].copy()
print("Final dataset shape:", df.shape)

# check duplicates
print(df.duplicated().sum())

#### Create word clouds

In [ ]:
def show_wordcloud(sentiment):
    text = " ".join(df[df["sentiment"] == sentiment]["cleaned_review"])
    phone_mask = np.array(Image.open('smartphone.jpg'))
    
    wordcloud = WordCloud(
        width=900,
        height=450,
        background_color="white",
        colormap="viridis",
        mask=phone_mask
    ).generate(text)

    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Most Frequent Words in {sentiment} Reviews")
    plt.show()

show_wordcloud("Positive")
show_wordcloud("Neutral")
show_wordcloud("Negative")


#### Label Encoding

In [ ]:
X=df['cleaned_review']
y=df['sentiment']
X.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(df["sentiment"])

print("Classes:", label_encoder.classes_)
print("Encoded labels:", y[:10])

#### splitting data into training and testing data

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify= y)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
df.to_csv("cleaned_sentiment_reviews.csv", index=False)

In [ ]:
df1 = pd.read_csv("cleaned_sentiment_reviews.csv")
df1.head(3)

### 03. Feature Extraction

In [ ]:
# Feature extraction using TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)

In [ ]:
import pandas as pd

# Step 1: Extract feature names from TF-IDF
feature_names = tfidf.get_feature_names_out()

# Step 2: Get vector for the first document
doc_vector = X_train_tfidf[0].toarray().flatten()

# Step 3: Pair feature names with their TF-IDF scores
doc_features = list(zip(feature_names, doc_vector))

# Step 4: Sort by score (descending) and take top 20
top_features = sorted(doc_features, key=lambda x: x[1], reverse=True)[:20]

# Step 5: Convert to DataFrame for table view
top_df = pd.DataFrame(top_features, columns=["Feature", "TF-IDF Score"])

print(top_df)

### 04. Model Building

#### Importing Models 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

#### Create the models

In [ ]:
# 1.Logistic Regression
logistic_model = LogisticRegression(max_iter=1000)

# Naive_bayes
naive_bayes_model = MultinomialNB()

# SVM
svm_model = LinearSVC()

# Random Forest
random_forest_model = RandomForestClassifier(n_estimators=100,random_state=42)

#XGBOOST
xgboost_model = XGBClassifier(n_estimators=100,random_state=42,eval_metric='logloss')

In [ ]:
# Train Logistic Regression
logistic_model.fit(X_train_tfidf, y_train)
y_pred_lr = logistic_model.predict(X_test_tfidf)


# Train Naive Bayes
naive_bayes_model.fit(X_train_tfidf, y_train)
y_pred_nb = naive_bayes_model.predict(X_test_tfidf)


# Train SVM
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)


# Train Random Forest
random_forest_model.fit(X_train_tfidf, y_train)
y_pred_rf = random_forest_model.predict(X_test_tfidf)


# Train XGBoost
xgboost_model.fit(X_train_tfidf, y_train)
y_pred_xgb = xgboost_model.predict(X_test_tfidf)


print("All 5 models trained and predictions generated successfully.")

#### Evaluating the Models 

In [ ]:
from sklearn.metrics import (confusion_matrix,classification_report,accuracy_score,precision_score,recall_score,f1_score)

In [ ]:
def eval_clf(y, yhat):

    # Confusion Matrix
    cm = confusion_matrix(y, yhat)

    plt.figure(figsize=(5, 4))

    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
        xticklabels=['Negative', 'Neutral', 'Positive'],
        yticklabels=['Negative', 'Neutral', 'Positive'])

    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')

    plt.show()


    # Classification Report
    print("Classification Report:")
    print(classification_report(y,yhat,target_names=['Negative', 'Neutral', 'Positive']))


    # Evaluation Metrics
    precision = precision_score(y, yhat, average='weighted')
    recall = recall_score(y, yhat, average='weighted')
    accuracy = accuracy_score(y, yhat)
    f1 = f1_score(y, yhat, average='weighted')


    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"F1-Score:  {f1:.4f}")

In [ ]:
print('Evaluation for Logistic Regression')
eval_clf(y_test, y_pred_lr)

In [ ]:
print('Evaluation for Naive Bayes')
eval_clf(y_test, y_pred_nb)

In [ ]:
print('Evaluation for Linear SVM')
eval_clf(y_test, y_pred_svm)

In [ ]:
print('Evaluation for Random Forest')
eval_clf(y_test, y_pred_rf)

In [ ]:
print('Evaluation for XGBoost')
eval_clf(y_test, y_pred_xgb)

In [ ]:
predictions = {
    "Logistic Regression": y_pred_lr,
    "Naive Bayes": y_pred_nb,
    "SVM": y_pred_svm,
    "Random Forest": y_pred_rf,
    "XGBoost": y_pred_xgb
}

results = []

for model_name, y_pred in predictions.items():

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    results.append([
        model_name,
        accuracy,
        precision,
        recall,
        f1
    ])

results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score"]
)

print(results_df)

In [ ]:
# Sort models by Accuracy
plot_df = results_df.sort_values("F1-Score")

plt.figure(figsize=(6, 3))

plt.barh(
    plot_df["Model"],
    plot_df["Accuracy"]
)

plt.xlabel("Accuracy")
plt.ylabel("Model")
plt.title("Accuracy Comparison of Sentiment Analysis Models")
plt.xlim(0, 1)

# Add values on bars
for i, value in enumerate(plot_df["Accuracy"]):
    plt.text(value + 0.01, i, f"{value:.3f}", va="center")

plt.tight_layout()
plt.show()

In [ ]:
# Get XGBoost results
xgb_result = results_df[results_df["Model"] == "XGBoost"].iloc[0]

metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
scores = [xgb_result[m] for m in metrics]

plt.figure(figsize=(6, 4))

bars = plt.bar(metrics, scores)

plt.title("XGBoost Performance - Sentiment Analysis")
plt.ylabel("Score")
plt.ylim(0, 1)

# Display values on bars
for bar, score in zip(bars, scores):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        score + 0.02,
        f"{score:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

### 05. Model comparison and selection
Select the model with the highest weighted F1-score, while also checking the per-class classification report. Weighted averages can hide weak performance on the smaller Neutral class, so do not rely on accuracy alone.


In [ ]:
results_df = results_df.sort_values("F1-Score", ascending=False).reset_index(drop=True)
display(results_df.style.format({"Accuracy": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}", "F1-Score": "{:.3f}"}))

best_model_name = results_df.loc[0, "Model"]
print(f"Highest weighted F1-score: {best_model_name}")
print("Review the class-level report above, especially Neutral recall, before finalizing a model.")

### 06. Example predictions
The prediction function below uses the same preprocessing and TF-IDF vectorizer fitted on the training data. These examples are illustrative; their outputs depend on the notebook run.


In [ ]:
# Predicting  statements using XGBOOST Model 
def predict_sentiment(review):

    # Preprocess the new review
    cleaned_review = preprocess_text(review)

    # Convert text into TF-IDF
    review_tfidf = tfidf.transform([cleaned_review])

    # Predict using original XGBoost model
    prediction = xgboost_model.predict(review_tfidf)[0]

    # Convert numerical label back to sentiment
    sentiment = label_encoder.inverse_transform([prediction])[0]

    return sentiment

In [ ]:
review = "Horrible product Very disappointed with the overall performance from Samsung"

result = predict_sentiment(review)

print("Review:", review)
print("Predicted Sentiment:", result)

In [ ]:
reviews = [
    "The product is excellent and I love it.",
    "The product is worst and completely useless.",
    "The product is okay, nothing special and its a average."
]

for review in reviews:
    print(review)
    print("Prediction:", predict_sentiment(review))
    print()

In [ ]:
for i in range(10):

    review = X_test.iloc[i]

    prediction = xgboost_model.predict(
        tfidf.transform([review])
    )[0]

    sentiment = label_encoder.inverse_transform([prediction])[0]

    print("Review:", review)
    print("Actual:", label_encoder.inverse_transform([y_test[i]])[0])
    print("Predicted:", sentiment)
    print("-" * 100)

In [ ]:
import joblib

joblib.dump(logistic_model, "logistic_model.pkl")
joblib.dump(naive_bayes_model, "naive_bayes_model.pkl")
joblib.dump(svm_model, "svm_model.pkl")
joblib.dump(random_forest_model, "random_forest_model.pkl")
joblib.dump(xgboost_model, "xgboost_model(1).pkl")

joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("All models and preprocessing files saved successfully!")

### 07. Deployment notes
The Streamlit app uses the XGBoost model, TF-IDF vectorizer, and label encoder saved by this notebook. Run the notebook from top to bottom before deploying so all artifacts are generated from the same training run.

Required files beside `app.py`:
- `xgboost_model(1).pkl`
- `tfidf_vectorizer.pkl`
- `label_encoder.pkl`

Install the project dependencies in the environment used for training and deployment. Keep scikit-learn and XGBoost versions compatible with the versions used to save the pickle files.

**Known limitation:** the earlier evaluation showed low recall for the Neutral class. Consider collecting more neutral examples, checking label quality, and tuning class weights/model parameters before presenting this as a production-ready classifier.
